In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
import pickle
from astropy.constants import e, h, c
from scipy.interpolate import interp1d
import pandas as pd
from tcbp.reduction.dataset import get_calib_path

from tcbp.analysis.lsst import LSSTRun

%matplotlib widget

In [ ]:
def plot_Qphotodiodes(run):
    """
    Compute and plot charges ratio between sphere and receiver photodiodes for one run

    run : LSSTRun
        LSSTRun object containing the data for the run
    Inv_electrometers : bool
        If True, the electrometers are inverted (this is the case for the ZTF and Stardice runs)
    """

    ### Get data
    wls = run["set_wl"]
    
    Q_sphere = run['pd_charge_per_burst'][wls > 0.]
    Q_receiver = run['calib_charge_per_burst'][wls > 0.]

    Q_sphere_err = run['pd_charge_per_burst_err'][wls > 0.]
    Q_receiver_err = run['calib_charge_per_burst_err'][wls > 0.]

    ratio = np.abs(Q_receiver / Q_sphere)
    ratio_err = np.abs(ratio) * np.sqrt((Q_sphere_err / Q_sphere)**2 + (Q_receiver_err / Q_receiver)**2)

    ### Plot
    fig, (ax1,ax2,ax3) = plt.subplots(3, sharex="all", figsize=(8,8))

    ax1.errorbar(wls[wls > 0.], Q_sphere, yerr=Q_sphere_err, c='b', marker='.', ls="")
    ax1.set_ylabel('Charges Sphere [C]')
        
    ax2.errorbar(wls[wls > 0.], Q_receiver, yerr=Q_receiver_err, c='r', marker='.', ls="")
    ax2.set_ylabel('Charges Receiver [C]')

    ax3.errorbar(wls[wls > 0.], ratio, yerr=ratio_err, c='indigo', marker='.', ls="")
    ax3.set_ylabel('Ratio Receiver / Sphere [No unit]')
    ax3.set_xlabel('wavelengths [nm]')

    fig.tight_layout()
    plt.show()

    return ratio, ratio_err

### Ratio Receiver / Sphere 

In [ ]:
run = np.load(data_dir + "run.npy", allow_pickle=True)
wls_run = run["set_wl"][run["set_wl"] > 300.]
ratio, ratio_err = plot_Qphotodiodes(run)

print(np.mean(np.abs(ratio_err) / ratio))


## Thierry and Kelian data

In [ ]:
### Thierry's data
tcbp = np.load(get_calib_path("cbp_laser_response_dr5.npy"), allow_pickle=True)
tcbp_func = interp1d(tcbp['cbp_wl'], tcbp['cbp_tr'], bounds_error=False, fill_value="extrapolate")
tcbp_err_func = interp1d(tcbp['cbp_wl'], tcbp['cbp_tr_err'], bounds_error=False, fill_value="extrapolate")

### Kelian's data
solarcell = np.loadtxt(get_calib_path("SC_QE_NewCell_20220401_LongQECurve-2_reduced_filtered.txt"), skiprows=1, delimiter=",")
pd_df_enhanced = pd.read_feather(get_calib_path("tCBP_v0_pd_df_fit.feather"))
sc_df_enhanced = pd.read_feather(get_calib_path("tCBP_v0_sc_df_fit.feather"))

sigma_A_SC = sc_df_enhanced['sigma_amplitude']
A_SC = sc_df_enhanced['amplitude']

sigma_A_PD = pd_df_enhanced['sigma_amplitude']
A_PD = pd_df_enhanced['amplitude']

T_CBP = A_SC/A_PD/np.interp(sc_df_enhanced['wl'], solarcell[:,0], solarcell[:,1])
T_CBP_wl = sc_df_enhanced['wl']

sigma_T_CBP = T_CBP * np.sqrt((sigma_A_SC/A_SC)**2 + (sigma_A_PD/A_PD)**2)


## Photodiode efficiencies

In [ ]:
wls_interp = np.arange(300, 1150, 4)
data_dir2 = '/home/lmousset/Projets_Recherche/LSST/Mesure_transmission_tcbp/'

# NIST photodiode calibration data
NIST_e_per_photon = np.load(data_dir2 + 'NIST_e_per_photon.npy', allow_pickle=True)
interpolator_pd_NIST_e_per_photon = interp1d(NIST_e_per_photon[0, :], NIST_e_per_photon[1, :], kind='cubic', fill_value='extrapolate')
interpolator_pd_NIST_e_per_photon_err = interp1d(NIST_e_per_photon[0, :], NIST_e_per_photon[2, :], kind='linear', fill_value='extrapolate')

pd_NIST_interp_e_per_photon = interpolator_pd_NIST_e_per_photon(wls_interp)
pd_NIST_interp_e_per_photon_err = interpolator_pd_NIST_e_per_photon_err(wls_interp)

# Marc photodiode data
pd8D057_Qeff = np.load(data_dir2 + 'photodiode_Qeff_8D057.npy', allow_pickle=True)
interpolator_pd8D057_Qeff = interp1d(pd8D057_Qeff[0, :], pd8D057_Qeff[1, :], kind='cubic', fill_value='extrapolate')
interpolator_pd8D057_Qeff_err = interp1d(pd8D057_Qeff[0, :], pd8D057_Qeff[2, :], kind='linear', fill_value='extrapolate')

pd8D057_interp_Qeff = interpolator_pd8D057_Qeff(wls_interp)
pd8D057_interp_Qeff_err = interpolator_pd8D057_Qeff_err(wls_interp)

plt.figure(figsize=(10, 6))
plt.errorbar(wls_interp, pd_NIST_interp_e_per_photon, yerr=pd_NIST_interp_e_per_photon_err, label='NIST photodiode calibration', c='r', marker='.', ls="")
plt.errorbar(wls_interp, pd8D057_interp_Qeff, yerr=pd8D057_interp_Qeff_err, label='Marc photodiode 8D057', c='g', marker='.', ls="")
plt.legend()
plt.xlabel('Wavelength [nm]')


## Lens transmission

In [ ]:
### Elana lens measurement
#with open(data_dir2 + 'Interp_lens_transmission.pkl', 'rb') as file:
 #  interpolator_lens = pickle.load(file)
#transmission_lens_interp = interpolator_lens(wls_run)


### New lens transmission data
with open('/home/lmousset/Projets_Recherche/LSST/tCBP_optical_bench/lens_transmission_ratio_dict.pkl', 'rb') as file:
    transmission_dict = pickle.load(file)

the_scans = ['20260703_163504', '20260706_195018', '20260707_174705', '20260707_193623']


plt.figure(figsize=(10,5))
for key in the_scans:
    print(f"Scan: {key}")
    ratio = transmission_dict[key][3]
    ratio_err = transmission_dict[key][4]
    wls = transmission_dict[key][2]

    plt.errorbar(wls, ratio, yerr=ratio_err, marker='.', ls="")
plt.ylim(0, 1)
plt.title(f"Transmission ratio for scan {key}")
plt.xlabel("Wavelength [nm]")
plt.ylabel("Transmission ratio [No unit]")
plt.grid()
plt.show()

print(transmission_dict.keys())



In [ ]:
# Calcul de la transmission moyenne

# Vérifier les wavelengths disponibles pour chaque scan
print("Wavelengths disponibles pour chaque scan:")
for key in the_scans:
    wls = transmission_dict[key][2]
    print(f"{key}: {len(wls)} points, de {wls.min():.1f} à {wls.max():.1f} nm")

# Trouver TOUTES les wavelengths uniques de tous les scans
all_unique_wls = np.array([])
for key in the_scans:
    all_unique_wls = np.union1d(all_unique_wls, transmission_dict[key][2])

print(f"\nToutes les wavelengths uniques: {len(all_unique_wls)} points, de {all_unique_wls.min():.1f} à {all_unique_wls.max():.1f} nm")

# Pour chaque wavelength, compter combien de scans l'ont et la moyenner
avg_ratio = []
avg_ratio_err = []
count_per_wl = []

for wl in all_unique_wls:
    ratios = []
    ratio_errs = []
    
    for key in the_scans:
        wls = transmission_dict[key][2]
        ratio = transmission_dict[key][3]
        ratio_err = transmission_dict[key][4]
        
        # Chercher si cette wavelength existe dans ce scan
        idx = np.where(np.isclose(wls, wl, rtol=1e-9))[0]
        if len(idx) > 0:
            ratios.append(ratio[idx[0]])
            ratio_errs.append(ratio_err[idx[0]])
    
    count = len(ratios)
    if count > 0:
        avg_ratio.append(np.mean(ratios))
        
        if count > 1:
            # Calculer le STD sur les scans
            err = np.std(ratios, ddof=1) / np.sqrt(count)
        else:
            # Un seul scan : garder l'erreur du dictionnaire
            err = ratio_errs[0]
        
        avg_ratio_err.append(err)
        count_per_wl.append(count)

avg_ratio = np.array(avg_ratio)
avg_ratio_err = np.array(avg_ratio_err)
count_per_wl = np.array(count_per_wl)

print(f"\nMoyenne complète: {len(avg_ratio)} points")
print(f"  - {np.sum(count_per_wl == 4)} points avec 4 scans")
print(f"  - {np.sum(count_per_wl == 3)} points avec 3 scans")
print(f"  - {np.sum(count_per_wl == 2)} points avec 2 scans")
print(f"  - {np.sum(count_per_wl == 1)} point(s) avec 1 scan")
print(f"Ratio moyen: {avg_ratio.min():.3f} à {avg_ratio.max():.3f}")

# Tracer
plt.figure(figsize=(12,6))
for key in the_scans:
    wls = transmission_dict[key][2]
    ratio = transmission_dict[key][3]
    ratio_err = transmission_dict[key][4]
    plt.errorbar(wls, ratio, yerr=ratio_err, marker='.', ls="", alpha=0.5, label=key)

plt.errorbar(all_unique_wls, avg_ratio, yerr=avg_ratio_err, 
                     marker='.', ls="", markersize=5, color='k', 
                     label=f'Moyenne')

plt.xlabel("Wavelength [nm]")
plt.ylabel("Transmission ratio")
plt.legend(fontsize=8)
plt.grid()
plt.show()

In [ ]:
new_interpolator_lens = interp1d(all_unique_wls, avg_ratio, kind='cubic', fill_value='extrapolate')
new_interpolator_lens_err = interp1d(all_unique_wls, avg_ratio_err, kind='linear', fill_value='extrapolate')

transmission_lens_interp = new_interpolator_lens(wls_interp)
transmission_lens_interp_err = new_interpolator_lens_err(wls_interp)

# Get several runs

In [ ]:
data_dir_list = [#main_repo + '20260627_213502_optical_bench_pinhole5mm_slit20/',        # Run 0 Tout premier run (colle avec les runs 4-5-6)
                 #main_repo + '20260628_193214_optical_bench_pinhole5mm_slit20/',        # Run 1 Papier écrasé
                 #main_repo + '20260629_134742_optical_bench_pinhole5mm_slit20/',        # Run 2 Papier écrasé (sampling max)
                 #main_repo + '20260629_152137_optical_bench_pinhole5mm_slit20/',        # Run 3 Papier écrasé (sampling max)
                 main_repo + '20260630_195101_optical_bench_pinhole5mm_slit20/',        # Run 4 Après avoir refait le foyer sur le vert
                 main_repo + '20260701_000936_optical_bench_pinhole5mm_slit20/',        # Run 5 Idem que Run 4
                 main_repo + '20260630_175411_optical_bench_pinhole5mm_slit20/',        # Run 6 Foyer sur le bleu 
                 #main_repo + '20260628_214053_optical_bench_pinhole1mm_slit20/',        # Run 7 Trou de 1mm
                 #main_repo + '20260629_004614_optical_bench_pinhole200um_slit20/',      # Run 8 Trou de 200µm, bruité mais ok
                 #main_repo + '20260629_170826_optical_bench_pinhole5mm_ZTFlike_slit20/',# Run 9 ZTF-like
                 #main_repo + '20260629_194844_optical_bench_pinhole200um_ZTFlike_slit20/', # Run 10 ZTF-like 200µm
                 #main_repo + '20260629_213351_optical_bench_pinhole5mm_SDlike_slit20/',  # Run 11 StarDice-like
                 #main_repo + '20260701_140913_optical_bench_pinhole5mm_ZTFlike_slit20/',  # Run 12 ZTF-like
                 #main_repo + '20260708_002717_optical_bench_pinhole5mm_8D057_slit20/',    # Run 13 Marc's photodiode 8D057
                 #main_repo + '20260708_013122_optical_bench_pinhole5mm_8D057_SDLike_slit20/' # Run 14 Marc's photodiode 8D057 StarDice-like
                 ]

nruns = len(data_dir_list)

all_ratio = []
all_ratio_err = []
all_wls_run = []
for i, data_dir in enumerate(data_dir_list):
    print(i)

    run = np.load(data_dir + "run.npy", allow_pickle=True)

    wls_run = run["set_wl"][run["set_wl"] > 0.]
    print(wls_run)
    ratio, ratio_err = plot_Qphotodiodes(run)

    all_wls_run.append(wls_run)
    all_ratio.append(ratio)
    all_ratio_err.append(ratio_err)

    print(np.mean(np.abs(ratio_err) / ratio))

In [ ]:
# Sort by wavelength
all_wls_run_sorted = []
for i in range(nruns):
    sort_indices = np.argsort(all_wls_run[i])
    all_wls_run_sorted.append(all_wls_run[i][sort_indices])
    all_ratio[i] = all_ratio[i][sort_indices]
    all_ratio_err[i] = all_ratio_err[i][sort_indices]

In [ ]:
## Plot all ratios
plt.figure(figsize=(10,5))
for i in range(nruns):
    print(i)
    #norm = all_ratio[i][all_wls_run_sorted[i]==769.996][0]
    norm = 1.#all_ratio[i][all_wls_run_sorted[i]==500.][0]
    print(norm)
    plt.errorbar(all_wls_run_sorted[i], all_ratio[i] / norm, yerr=np.abs(all_ratio_err[i] / norm), marker='.', ls="", label=f'Run {i}')
plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere')
plt.grid(True)
#plt.ylim(0, 2.5)
plt.legend()

In [ ]:
# Rapport entre 2 scans
irun1 = 2
irun2 = 1

commun_wls = np.intersect1d(all_wls_run_sorted[irun1], all_wls_run_sorted[irun2])
print(f"Commun wavelengths between run {irun1} and run {irun2}: {commun_wls}")

# Utiliser searchsorted pour maintenir l'ordre correct des longueurs d'onde
indices1 = np.searchsorted(all_wls_run_sorted[irun1], commun_wls)
indices2 = np.searchsorted(all_wls_run_sorted[irun2], commun_wls)

commun_ratio1 = all_ratio[irun1][indices1]
commun_ratio2 = all_ratio[irun2][indices2]

commun_ratio_err1 = all_ratio_err[irun1][indices1]
commun_ratio_err2 = all_ratio_err[irun2][indices2]

rapport = commun_ratio1 / commun_ratio2
rapport_err = np.sqrt((commun_ratio_err1 / commun_ratio1)**2 + (commun_ratio_err2 / commun_ratio2)**2) * rapport


plt.figure(figsize=(10,5))
plt.errorbar(commun_wls, rapport, yerr=rapport_err, c='orange', marker='.', ls="")
#plt.ylim(0.99, 1.01)
plt.ylabel('ratio entre deux runs')

## Apply the correction factor : Photodiode and Lens 

In [ ]:
### Correction factor
all_ratio_corrected = []
all_ratio_corrected_err = []
for i in range(nruns):
    transmission_lens_interp = new_interpolator_lens(all_wls_run_sorted[i]) # New lens measurement
    transmission_lens_interp_err = new_interpolator_lens_err(all_wls_run_sorted[i]) # New lens measurement
    #transmission_lens_interp = interpolator_lens(all_wls_run_sorted[i]) # Elana measurement

    if i in [4, 5]:
        print(f"Run {i} is using 8D057 photodiode")
        pd_interp_e_per_photon = interpolator_pd8D057_Qeff(all_wls_run_sorted[i])
        pd_interp_e_per_photon_err = interpolator_pd8D057_Qeff_err(all_wls_run_sorted[i])
    
    else :
        print(f"Run {i} is using NIST photodiode")
        pd_interp_e_per_photon = interpolator_pd_NIST_e_per_photon(all_wls_run_sorted[i])
        pd_interp_e_per_photon_err = interpolator_pd_NIST_e_per_photon_err(all_wls_run_sorted[i])

    correction_factor = 1 / (pd_interp_e_per_photon * transmission_lens_interp * e.value) 
    ratio_corrected = all_ratio[i] * correction_factor

    ### Erreurs
    ratio_corrected_err = np.sqrt((all_ratio_err[i]/all_ratio[i])**2 + 
                                  (pd_interp_e_per_photon_err / pd_interp_e_per_photon)**2 + 
                                  (transmission_lens_interp_err / transmission_lens_interp)**2) * ratio_corrected
    
    all_ratio_corrected.append(ratio_corrected)
    all_ratio_corrected_err.append(ratio_corrected_err)

In [ ]:
plt.figure(figsize=(10,5))
for i in range(nruns):
    #norm = all_ratio_corrected[i][all_wls_run_sorted[i]==500.][0]
    #print(norm)
    plt.errorbar(all_wls_run_sorted[i], all_ratio_corrected[i], yerr=np.abs(all_ratio_corrected_err[i]), marker='.', ls="-", label=f'Run {i}')
plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere corrected')
plt.grid(True)
plt.legend()

In [ ]:
# Moyenne des runs

# Trouver TOUTES les wavelengths uniques de tous les runs
all_unique_wls_runs = np.array([])
for i in range(nruns):
    all_unique_wls_runs = np.union1d(all_unique_wls_runs, all_wls_run_sorted[i])

print(f"Toutes les wavelengths uniques des runs: {len(all_unique_wls_runs)} points, de {all_unique_wls_runs.min():.1f} à {all_unique_wls_runs.max():.1f} nm")

# Pour chaque wavelength, compter combien de runs l'ont et la moyenner
avg_ratio_corrected = []
avg_ratio_corrected_err = []
count_per_wl_runs = []

for wl in all_unique_wls_runs:
    ratios = []
    ratio_errs = []
    
    for i in range(nruns):
        wls = all_wls_run_sorted[i]
        ratio = all_ratio_corrected[i]
        ratio_err = all_ratio_corrected_err[i]
        
        # Chercher si cette wavelength existe dans ce run
        idx = np.where(np.isclose(wls, wl, rtol=1e-9))[0]
        if len(idx) > 0:
            ratios.append(ratio[idx[0]])
            ratio_errs.append(ratio_err[idx[0]])
    
    count = len(ratios)
    if count > 0:
        avg_ratio_corrected.append(np.mean(ratios))
        
        if count > 1:
            # Calculer le STD sur les runs
            err = np.std(ratios, ddof=1) / np.sqrt(count)
        else:
            # Un seul run : garder l'erreur du run
            err = ratio_errs[0]
        
        avg_ratio_corrected_err.append(err)
        count_per_wl_runs.append(count)

avg_ratio_corrected = np.array(avg_ratio_corrected)
avg_ratio_corrected_err = np.array(avg_ratio_corrected_err)
count_per_wl_runs = np.array(count_per_wl_runs)

print(f"\nMoyenne des runs: {len(avg_ratio_corrected)} points")
print(f"  - {np.sum(count_per_wl_runs == nruns)} points avec {nruns} runs")
for n in range(nruns-1, 0, -1):
    count_n = np.sum(count_per_wl_runs == n)
    if count_n > 0:
        print(f"  - {count_n} point(s) avec {n} run(s)")
print(f"Ratio moyen: {avg_ratio_corrected.min():.3f} à {avg_ratio_corrected.max():.3f}")

# Tracer
plt.figure(figsize=(12,6))
for i in range(nruns):
    plt.errorbar(all_wls_run_sorted[i], all_ratio_corrected[i], yerr=np.abs(all_ratio_corrected_err[i]), 
                 marker='.', ls="", alpha=0.5, label=f'Run {i}')

plt.errorbar(all_unique_wls_runs, avg_ratio_corrected, yerr=avg_ratio_corrected_err, 
             marker='o', ls="-", linewidth=2, markersize=5, color='k', 
             label=f'Moyenne des {nruns} runs')

plt.xlabel("Wavelength [nm]")
plt.ylabel("Transmission ratio (corrected)")
plt.legend(fontsize=8)
plt.grid()
plt.show()

In [ ]:
# Rapport entre 2 scans après avoir appliqué le facteur de correction
irun1 = 2
irun2 = 1

commun_wls = np.intersect1d(all_wls_run_sorted[irun1], all_wls_run_sorted[irun2])
print(f"Commun wavelengths between run {irun1} and run {irun2}: {commun_wls}")

# Utiliser searchsorted pour maintenir l'ordre correct des longueurs d'onde
indices1 = np.searchsorted(all_wls_run_sorted[irun1], commun_wls)
indices2 = np.searchsorted(all_wls_run_sorted[irun2], commun_wls)

commun_ratio1 = all_ratio_corrected[irun1][indices1]
commun_ratio2 = all_ratio_corrected[irun2][indices2]

commun_ratio_err1 = all_ratio_corrected_err[irun1][indices1]
commun_ratio_err2 = all_ratio_corrected_err[irun2][indices2]

rapport = commun_ratio1 / commun_ratio2
rapport_err = np.sqrt((commun_ratio_err1 / commun_ratio1)**2 + (commun_ratio_err2 / commun_ratio2)**2) * rapport


plt.figure(figsize=(10,5))
plt.errorbar(commun_wls, rapport, yerr=np.abs(rapport_err), c='orange', marker='.', ls="")
#plt.ylim(0.9, 1.05)
plt.ylabel('ratio entre deux runs')

In [ ]:
# Save a run
irun = 2
my_run = np.zeros((3, len(all_wls_run_sorted[irun])))
my_run[0,:] = all_wls_run_sorted[irun]
my_run[1,:] = all_ratio_corrected[irun]
my_run[2,:] = all_ratio_corrected_err[irun]

# Transformer ce tableau en record array
np.save(data_dir_list[irun] + "run_corrected_new_lens_transmission.npy", np.core.records.fromarrays(my_run, names='cbp_wl, cbp_tr, cbp_tr_err', formats='f8,f8,f8'))
